# Học Machine Learning: Logistic Regression

Chào mừng bạn đến với notebook học **Logistic Regression** – một thuật toán phân loại có giám sát (supervised classification) được sử dụng rộng rãi để dự đoán xác suất của biến mục tiêu nhị phân. Trong notebook này, chúng ta sẽ đi từ lý thuyết nền tảng (hàm Sigmoid, Logit, Log Loss), cách sử dụng thư viện scikit-learn, tiền xử lý dữ liệu, đến đánh giá mô hình bằng các metric chuyên sâu. Bạn sẽ được thực hành trên bộ dữ liệu **Heart Failure Prediction** để dự đoán khả năng mắc bệnh tim mạch.

## Chuẩn bị dữ liệu

**Tên dataset:** Heart Failure Prediction Dataset
**Nguồn:** Kaggle (fedesoriano/heart-failure-prediction)
**Kích thước:** 918 dòng, 12 cột (11 đặc trưng lâm sàng + 1 nhãn)
**Biến mục tiêu:** `HeartDisease` (0 = không bệnh, 1 = có bệnh tim mạch)

**Đặc trưng chính:** Age, Sex, ChestPainType, RestingBP, Cholesterol, FastingBS, RestingECG, MaxHR, ExerciseAngina, Oldpeak, ST_Slope.

**Tiền xử lý:** Các giá trị 0 bất hợp lý (Cholesterol, RestingBP) được coi là khuyết và impute bằng median. Dữ liệu đã được chuẩn hóa bằng StandardScaler và chia train/test sẵn.

### 1. Load Dataset & Inspection

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

relative_data_path = Path("notebookforge/datasets/heart.csv")
data_candidates = [
    Path.cwd() / relative_data_path,
    Path.cwd() / "datasets" / relative_data_path.name,
    Path.cwd().parent / "datasets" / relative_data_path.name,
]
data_path = next(
    (path.resolve() for path in data_candidates if path.is_file()),
    Path.cwd() / relative_data_path,
)

# Kiểm tra sự tồn tại của Dataset File
if not data_path.is_file():
    raise FileNotFoundError(
        f"❌ KHÔNG TÌM THẤY DATASET: 'Heart Failure Prediction' tại đường dẫn '{os.path.abspath(data_path)}'.\n"
        f"👉 Vui lòng đảm bảo bạn đã copy file CSV vào đúng thư mục 'notebookforge/datasets/'!"
    )

# Load Heart Failure Prediction Dataset
df = pd.read_csv(data_path)

print(f"Dataset Successfully Loaded! Shape: {df.shape}")
df.head()

### 2. EDA & Handling Missing/Outlier Values

In [ ]:
# 1. Loại bỏ dòng vô lý RestingBP = 0
df = df.copy()
df['RestingBP'] = df['RestingBP'].replace(0, np.nan)

# 2. Chuyển Cholesterol = 0 thành NaN để Impute (Dữ liệu khuyết ngầm y khoa)
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)

print("Kiểm tra giá trị Null trước khi chia dữ liệu:")
print(df.isnull().sum())

### 3. Feature Encoding, Scaling & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. One-Hot Encoding cho các biến Categorical (Sex, ChestPainType, RestECG, ExerciseAngina, ST_Slope)
X = pd.get_dummies(df.drop(columns=['HeartDisease']), drop_first=True)
y = df['HeartDisease']

# 2. Chia Train/Test (Stratify theo nhãn)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit imputer/scaler chỉ trên train để tránh data leakage.
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ X_train shape: {X_train_scaled.shape}, X_test shape: {X_test_scaled.shape}")

## Module m1: Lý thuyết nền tảng: Sigmoid, Logit và Log Loss

**Mục tiêu:** Hiểu bản chất của Logistic Regression: cách chuyển tổ hợp tuyến tính thành xác suất qua hàm Sigmoid, khái niệm Logit (odds ratio) và hàm mất mát Log Loss dùng để tối ưu.

### Hàm Sigmoid

Hàm Sigmoid có công thức: $\sigma(z) = \frac{1}{1 + e^{-z}}$. Nó nhận đầu vào là một số thực $z$ (tổ hợp tuyến tính của các đặc trưng) và trả về một giá trị trong khoảng (0, 1), thể hiện xác suất dự đoán. Khi $z$ càng lớn, xác suất tiến gần về 1; khi $z$ càng nhỏ, xác suất tiến gần về 0. Tại $z=0$, xác suất bằng 0.5.

### Logit (Log-odds)

Logit là hàm ngược của Sigmoid, được định nghĩa là $\text{logit}(p) = \ln\left(\frac{p}{1-p}\right)$. Nó biểu diễn log của tỷ số odds (xác suất xảy ra / xác suất không xảy ra). Trong Logistic Regression, mô hình học các hệ số sao cho tổ hợp tuyến tính $z = w^T x + b$ chính là logit của xác suất dự đoán.

### Log Loss (Binary Cross-Entropy)

Log Loss là hàm mất mát dùng để tối ưu mô hình. Với mỗi mẫu có nhãn thật $y \in \{0, 1\}$ và xác suất dự đoán $p$, log loss được tính: $L = -[y \log(p) + (1-y) \log(1-p)]$. Hàm này phạt nặng khi mô hình tự tin nhưng sai, giúp quá trình huấn luyện hội tụ tốt hơn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demo: Vẽ đồ thị hàm Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_vals = np.linspace(-10, 10, 100)
p_vals = sigmoid(z_vals)

plt.figure(figsize=(6, 4))
plt.plot(z_vals, p_vals, label='Sigmoid function')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7)
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.title('Hàm Sigmoid')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Tính xác suất cho một giá trị z cụ thể
z_example = 2.5
p_example = sigmoid(z_example)
print(f'Sigmoid({z_example}) = {p_example:.4f}')

### Bài tập m1_ex1: Tính xác suất từ sigmoid

**Yêu cầu:** Cho một điểm dữ liệu có tổ hợp tuyến tính z = 2.5. Hãy tính xác suất dự đoán bằng công thức sigmoid và giải thích ý nghĩa của giá trị này.

In [ ]:
# TODO: Hoàn thành hàm sigmoid_exercise_1 và tính xác suất cho z = 2.5
# Gợi ý: định nghĩa hàm sigmoid_exercise_1(z) = 1 / (1 + exp(-z))

def sigmoid_exercise_1(z):
    # Trả về giá trị sigmoid_exercise_1 của z
    pass

m1_ex1_prob = None  # Gán kết quả sigmoid_exercise_1(2.5) vào biến này

# In kết quả
if m1_ex1_prob is not None:
    print(f'Xác suất dự đoán cho z=2.5 là: {m1_ex1_prob:.4f}')
else:
    print('Hãy hoàn thành bài tập trước khi xem kết quả.')

## Module m2: Cài đặt LogisticRegression trong scikit-learn

**Mục tiêu:** Nắm được cách sử dụng class `LogisticRegression` từ scikit-learn, các tham số chính (`penalty`, `solver`, `class_weight`) và các phương thức `fit`, `predict`, `predict_proba`.

### LogisticRegression trong scikit-learn

`LogisticRegression` là class chính để xây dựng mô hình hồi quy logistic. Các tham số quan trọng:
- **`penalty`**: Kỹ thuật regularization (L1, L2, elasticnet) giúp tránh overfitting. Mặc định là 'l2'.
- **`solver`**: Thuật toán tối ưu (lbfgs, liblinear, saga...). Với dữ liệu nhỏ, 'lbfgs' thường hoạt động tốt.
- **`class_weight`**: Xử lý mất cân bằng lớp, có thể đặt 'balanced' để tự động điều chỉnh trọng số.

Phương thức chính:
- `fit(X, y)`: Huấn luyện mô hình trên dữ liệu.
- `predict(X)`: Dự đoán nhãn (0 hoặc 1).
- `predict_proba(X)`: Trả về xác suất cho từng lớp, shape (n_samples, 2).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Demo: Huấn luyện mô hình LogisticRegression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Dự đoán và đánh giá
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy trên tập test: {accuracy:.4f}')

# Xác suất dự đoán cho 3 mẫu đầu tiên
proba_3 = model.predict_proba(X_test[:3])
print('Xác suất dự đoán cho 3 mẫu đầu tiên:')
print(proba_3)

# Kiểm tra hệ số của mô hình
print(f'Hệ số (coef): {model.coef_}')
print(f'Intercept: {model.intercept_}')

### Bài tập m2_ex1: Huấn luyện mô hình LogisticRegression

**Yêu cầu:** Sử dụng `LogisticRegression` từ scikit-learn, huấn luyện trên dữ liệu `X_train`, `y_train` (đã có sẵn). In ra accuracy trên tập test và xác suất dự đoán cho 3 mẫu đầu tiên.

In [ ]:
# TODO: Huấn luyện mô hình LogisticRegression
# Gợi ý: model = LogisticRegression(max_iter=1000); model.fit(X_train, y_train)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

m2_ex1_model = None  # Gán mô hình đã huấn luyện vào biến này
m2_ex1_accuracy = None  # Gán accuracy trên tập test
m2_ex1_proba = None  # Gán xác suất dự đoán cho 3 mẫu đầu tiên

# In kết quả
if m2_ex1_model is not None:
    print(f'Accuracy: {m2_ex1_accuracy:.4f}')
    print(f'Xác suất 3 mẫu đầu: {m2_ex1_proba}')
else:
    print('Hãy hoàn thành bài tập trước khi xem kết quả.')

## Module m3: Tiền xử lý dữ liệu: StandardScaler

**Mục tiêu:** Hiểu vì sao cần chuẩn hóa dữ liệu (StandardScaler) cho Logistic Regression và cách hệ thống đã tự động xử lý bước này.

### StandardScaler

`StandardScaler` là một kỹ thuật chuẩn hóa dữ liệu, biến đổi mỗi đặc trưng về phân phối chuẩn với mean = 0 và độ lệch chuẩn = 1. Công thức: $x_{scaled} = \frac{x - \mu}{\sigma}$.

**Tại sao cần chuẩn hóa?** Logistic Regression sử dụng gradient descent để tối ưu. Khi các đặc trưng có thang đo khác nhau (ví dụ: tuổi 20-80, cholesterol 100-400), quá trình hội tụ sẽ chậm và không ổn định. Chuẩn hóa giúp:
- Tăng tốc độ hội tụ của thuật toán.
- Cải thiện độ chính xác của mô hình.
- Giúp các hệ số có thể so sánh được với nhau.

Trong notebook này, dữ liệu đã được chuẩn hóa sẵn bằng `StandardScaler` trước khi chia train/test. Bạn có thể truy cập phiên bản đã chuẩn hóa qua biến `X_train_scaled` và `X_test_scaled` nếu cần.

In [ ]:
# Demo: Kiểm tra dữ liệu đã được chuẩn hóa
print('Kích thước X_train:', X_train.shape)
print('Kích thước X_test:', X_test.shape)

# Kiểm tra một vài giá trị thống kê của dữ liệu gốc
print('\nThống kê dữ liệu gốc (trước chuẩn hóa):')
print(X_train.iloc[:, :2].describe())

# Nếu có biến X_train_scaled, kiểm tra thống kê sau chuẩn hóa
if 'X_train_scaled' in globals():
    print('\nThống kê dữ liệu sau chuẩn hóa:')
    print(pd.DataFrame(X_train_scaled, columns=X_train.columns).iloc[:, :2].describe())
else:
    print('\nBiến X_train_scaled chưa được tạo (dữ liệu đã được chuẩn hóa sẵn trong X_train).')

## Module m4: Đánh giá mô hình phân loại

**Mục tiêu:** Sử dụng các metric đánh giá: Confusion Matrix, Precision, Recall, F1-Score, ROC-AUC để đánh giá chất lượng mô hình Logistic Regression.

### Các metric đánh giá

- **Confusion Matrix**: Bảng thể hiện số lượng True Positive (TP), True Negative (TN), False Positive (FP), False Negative (FN). Từ đó tính được các metric khác.
- **Precision**: Tỷ lệ dự đoán đúng trong số các mẫu được dự đoán là positive: $\text{Precision} = \frac{TP}{TP + FP}$.
- **Recall (Sensitivity)**: Tỷ lệ dự đoán đúng trong số các mẫu thực sự là positive: $\text{Recall} = \frac{TP}{TP + FN}$.
- **F1-Score**: Trung bình điều hòa của Precision và Recall: $F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$.
- **ROC-AUC**: Diện tích dưới đường cong ROC, thể hiện khả năng phân biệt giữa hai lớp. Giá trị càng gần 1 càng tốt.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Demo: Đánh giá mô hình đã huấn luyện ở module 2
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# In classification report
print('Classification Report:')
print(classification_report(y_test, y_pred))

# Tính ROC-AUC
roc_auc = roc_auc_score(y_test, y_proba)
print(f'ROC-AUC: {roc_auc:.4f}')

# Vẽ Confusion Matrix bằng matplotlib
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='black')
plt.show()

# Vẽ ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Bài tập m4_ex1: Tính các metric đánh giá

**Yêu cầu:** Dùng `classification_report` và `roc_auc_score` từ scikit-learn để tính Precision, Recall, F1-Score và ROC-AUC cho mô hình đã huấn luyện ở module 2. In ra kết quả.

In [ ]:
# TODO: Tính các metric đánh giá cho mô hình m2_ex1_model
# Gợi ý: dùng classification_report(y_test, y_pred) và roc_auc_score(y_test, y_proba)

from sklearn.metrics import classification_report, roc_auc_score

# Sử dụng m2_ex1_model đã huấn luyện ở bài tập m2_ex1
m4_ex1_y_pred = None  # Gán kết quả predict của m2_ex1_model
m4_ex1_y_proba = None  # Gán xác suất predict_proba của m2_ex1_model (cột 1)
m4_ex1_report = None  # Gán classification_report
m4_ex1_roc_auc = None  # Gán roc_auc_score

# In kết quả
if m4_ex1_report is not None:
    print(m4_ex1_report)
    print(f'ROC-AUC: {m4_ex1_roc_auc:.4f}')
else:
    print('Hãy hoàn thành bài tập trước khi xem kết quả.')

## Kiểm tra kết quả

Phần này dùng để kiểm tra tự động kết quả của bạn. Hãy chạy từng cell bên dưới sau khi hoàn thành bài tập tương ứng.

In [ ]:
try:
    # Kiểm tra m1_ex1
    if m1_ex1_prob is not None:
        assert abs(m1_ex1_prob - 0.9241) < 0.01, "Sigmoid(2.5) phải xấp xỉ 0.9241"
        print('✅ Bài tập m1_ex1 đúng!')
    else:
        print('⚠️ Hãy hoàn thành bài tập m1_ex1 trước khi kiểm tra.')
except (AssertionError, TypeError, ValueError, NameError, NotImplementedError) as exercise_error:
    print(f'Bài tập chưa hoàn thành: {exercise_error}')

In [ ]:
try:
    # Kiểm tra m2_ex1
    if m2_ex1_model is not None:
        assert hasattr(m2_ex1_model, 'predict_proba') and m2_ex1_model.coef_ is not None, "Mô hình phải có predict_proba và coef_"
        print('✅ Bài tập m2_ex1 đúng!')
    else:
        print('⚠️ Hãy hoàn thành bài tập m2_ex1 trước khi kiểm tra.')
except (AssertionError, TypeError, ValueError, NameError, NotImplementedError) as exercise_error:
    print(f'Bài tập chưa hoàn thành: {exercise_error}')

In [ ]:
try:
    # Kiểm tra m4_ex1
    if m4_ex1_roc_auc is not None:
        assert m4_ex1_roc_auc > 0.5, "ROC-AUC phải lớn hơn 0.5"
        print('✅ Bài tập m4_ex1 đúng!')
    else:
        print('⚠️ Hãy hoàn thành bài tập m4_ex1 trước khi kiểm tra.')
except (AssertionError, TypeError, ValueError, NameError, NotImplementedError) as exercise_error:
    print(f'Bài tập chưa hoàn thành: {exercise_error}')